# Step 7: Slack notifications

![Step 7 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/07-step.png)

## What you're building today

A ping on your phone the moment a bird is identified.

We'll hook up a Slack webhook. Every sighting gets posted to a Slack channel with the species name, the snapshot, and the timestamp.

By the end of this notebook you'll get a real Slack message on your phone when the loop detects a bird.

## Step 7.1 — What is a webhook?

A **webhook** is a URL that accepts incoming data. You POST a JSON message to it, and Slack shows it in a channel.

You create one in Slack:

1. Go to https://api.slack.com/apps → Create New App → From scratch
2. Pick a workspace + name (e.g. "Bird Watcher")
3. **Incoming Webhooks** → toggle ON → **Add New Webhook to Workspace**
4. Pick the channel (e.g. `#birds`)
5. Copy the webhook URL — it looks like `https://hooks.slack.com/services/T0.../B0.../abc123...`

For testing in this notebook, we'll use a placeholder. Replace it with your real webhook to send actual messages.

In [ ]:
import requests
import os
from pathlib import Path

# REPLACE THIS with your real webhook URL after you set up Slack.
# For now we leave it blank so nothing actually posts.
SLACK_WEBHOOK = os.environ.get("BIRD_WATCHER_WEBHOOK", "")

if SLACK_WEBHOOK:
    print("Webhook configured — notifications will be real.")
else:
    print("No webhook set yet — messages will be DRY-RUN (printed, not sent).")
    print("Set BIRD_WATCHER_WEBHOOK env var or edit SLACK_WEBHOOK above.")

## Step 7.2 — Format the message

Slack messages use a simple JSON format called "Block Kit". We build a small block with a header, a species line, and the snapshot.

In [ ]:
def format_sighting_message(species: str, confidence: float, snapshot_path: str) -> dict:
    """Build the Slack message body."""
    confidence_pct = int(confidence * 100)
    return {
        "blocks": [
            {
                "type": "header",
                "text": {"type": "plain_text", "text": f"New bird: {species}"}
            },
            {
                "type": "section",
                "fields": [
                    {"type": "mrkdwn", "text": f"*Species:*\n{species}"},
                    {"type": "mrkdwn", "text": f"*Confidence:*\n{confidence_pct}%"},
                ]
            },
            {
                "type": "context",
                "elements": [
                    {"type": "mrkdwn", "text": f"_snapshot: {snapshot_path}_"}
                ]
            }
        ]
    }

msg = format_sighting_message("American Robin", 0.92, "data/snapshots/r1.jpg")
import json
print(json.dumps(msg, indent=2))

## Step 7.3 — Send (or dry-run)

If a webhook is configured we POST. Otherwise we just print. Same code path either way.

In [ ]:
def post_sighting(species: str, confidence: float, snapshot_path: str) -> bool:
    """Post a sighting to Slack. Returns True if sent (or dry-run successful)."""
    msg = format_sighting_message(species, confidence, snapshot_path)

    if not SLACK_WEBHOOK:
        print(f"[DRY-RUN] would post: {species} ({confidence:.2f}) -> {snapshot_path}")
        return True

    try:
        r = requests.post(SLACK_WEBHOOK, json=msg, timeout=10)
        r.raise_for_status()
        print(f"[SENT] {species} ({confidence:.2f})")
        return True
    except Exception as e:
        print(f"[FAILED] {e}")
        return False

# Test with a fake sighting
post_sighting("Northern Cardinal", 0.87, "data/snapshots/test.jpg")

## Step 7.4 — Hook into the pipeline

Every time a bird is identified AND saved, also ping Slack. We re-use everything from issues 4-6.

In [ ]:
import time, sqlite3
from PIL import Image
from ultralytics import YOLO

RUNNING_IN_COLAB = "COLAB_GPU" in os.environ
SOURCE = (
    "https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/data/samples/sample-bird.jpg"
    if RUNNING_IN_COLAB
    else "http://192.168.1.42:8080/photo.jpg"
)
detector = YOLO("yolov8n.pt")

for i in range(2):
    frame_path = Path(f"data/snapshots/slack-{i:03d}.jpg")
    frame_path.write_bytes(requests.get(SOURCE, timeout=10).content)

    results = detector(frame_path, verbose=False)
    bird_boxes = [b for b in results[0].boxes if int(b.cls[0]) == 14 and float(b.conf[0]) > 0.4]

    if bird_boxes:
        box = bird_boxes[0]
        bbox = tuple(int(v) for v in box.xyxy[0])
        crop = Image.open(frame_path).crop(bbox)
        species = classifier(crop)[0]
        record_sighting(species["label"], species["score"], str(frame_path), bbox)
        post_sighting(species["label"], species["score"], str(frame_path))
    else:
        print(f"[{i+1}/2] no bird — nothing to alert on")

    if i < 1:
        time.sleep(3)

print("Done")

## Done?

If `SLACK_WEBHOOK` is set, you should have a real ping on your phone by now. 🎉

If not, the dry-run output should print the would-be messages — same content, just no Slack delivery. Set the webhook and re-run Step 7.4 to actually send.

## What's next

Issue **#8: Web UI hello world**. Slack is great for alerts, but for browsing what birds came by over the past week, a webpage is nicer. Next we build one. 🐦